# SHRIMP AI - PIPELINE HSV (Ambiente seco)

## Etapa 1: Configuração do Ambiente e Importação da Imagem
Nesta primeira etapa, preparamos o ambiente de desenvolvimento importando as bibliotecas essenciais para o Processamento Digital de Imagens (PDI): o **OpenCV** (para manipulação matemática das matrizes da imagem) e o **Matplotlib** (para exibição dos gráficos).

Também criamos a função auxiliar `mostrar()`. Ela resolve uma particularidade do OpenCV, que lê imagens no padrão de cores **BGR** (Azul, Verde, Vermelho), convertendo-as para o padrão universal **RGB** antes de exibir na tela, garantindo que as cores reais sejam mantidas. Por fim, habilitamos um assistente para fazer o upload do arquivo diretamente do computador.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files

In [ ]:
plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['image.cmap'] = 'gray'

def mostrar(img, titulo='', cmap=None):
    plt.figure()
    if img.ndim == 3:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    plt.title(titulo)
    plt.axis('off')
    plt.show()

print('ok')
uploaded = files.upload()
fname = next(iter(uploaded))
bgr_original = cv2.imread(fname)
print(f'imagem: {fname}  |  dimensoes: {bgr_original.shape[1]} x {bgr_original.shape[0]}')
mostrar(bgr_original, 'Foto original')

## Etapa 2: Conversão e Decomposição no Espaço de Cores HSV
Imagens comuns (RGB) misturam a informação de cor e iluminação nos mesmos canais. Para facilitar o isolamento das pós-larvas, convertemos a imagem para o espaço de cores **HSV**:
* **H (Hue / Matiz):** Representa a cor pura (o tom).
* **S (Saturation / Saturação):** Representa a vivacidade ou pureza da cor.
* **V (Value / Brilho):** Representa a intensidade de luz.

Separamos a imagem nesses três canais isolados. O canal **S (Saturação)** provou ser o melhor para o nosso objetivo, pois destaca com forte intensidade (tons mais claros) o corpo das pós-larvas e ignora fundos neutros ou variações de sombras (tons escuros).

In [ ]:
hsv = cv2.cvtColor(bgr_original, cv2.COLOR_BGR2HSV)
h, s, v = cv2.split(hsv)

mostrar(h, 'Canal H (Matiz / Hue)')
mostrar(s, 'Canal S (Saturação / Saturation)')
mostrar(v, 'Canal V (Valor-Brilho / Value)')

## Etapa 3: Binarização Automática com o Método de Otsu
Com o canal de Saturação selecionado, precisamos transformar a imagem em "preto e branco" absoluto (binária) para separar o que é larva do que é fundo.

Para não escolhermos um ponto de corte visualmente no "chute", aplicamos o algoritmo de **Otsu**. Ele analisa estatisticamente o histograma de distribuição dos pixels e calcula de forma automática o limiar (threshold) matemático ideal. Tudo o que for mais brilhante que esse limiar vira branco (valor 255, nossas larvas) e o que for mais escuro vira preto (valor 0, o fundo).

In [ ]:
limiar_calculado, imagem_binaria = cv2.threshold(
    s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

print(f'Otsu determinou o limiar ideal em: {limiar_calculado}')

mostrar(imagem_binaria, f'Resultado da Limiarização de Otsu (Limiar: {limiar_calculado})')

## Etapa 4 (Interativa/Opcional): Ajuste Fino e Filtragem de Componentes
Imagens biológicas frequentemente apresentam ruídos, como pequenas sujeiras na água ou reflexos. Nesta etapa interativa, criamos um filtro baseado em **Componentes Conexas** para calcular a área (em pixels) de cada elemento branco isolado.

Definimos um parâmetro experimental **$X$**: qualquer ponto menor que $X$ pixels é considerado ruído e apagado. Em seguida, aplicamos uma operação morfológica de **Dilatação** usando um elemento estruturante (matriz) de 3x3. A dilatação "engrossa" as bordas brancas, garantindo que partes de uma mesma larva que se separaram na binarização voltem a se unir antes da contagem.

In [ ]:
import cv2
import numpy as np
from ipywidgets import interact, IntSlider

# Função para processamento e ajuste fino dos parâmetros de filtragem e morfologia
def analise_interativa(X=10, iteracoes_dilatacao=2):
    # 1. Definir o elemento estruturante 3x3 para a dilatação
    kernel = np.ones((3,3), np.uint8)

    # 2. Identificar componentes conexas e suas estatísticas na imagem binarizada (Otsu)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(imagem_binaria, connectivity=8)

    # Criar uma imagem vazia para reconstruir apenas os objetos que atendem ao critério de tamanho
    img_filtrada = np.zeros_like(imagem_binaria)

    # Filtrar componentes com base na área (quantidade de pixels brancos)
    for i in range(1, num_labels): # O rótulo 0 é o fundo
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= X:
            img_filtrada[labels == i] = 255

    # 3. Aplicar a operação de dilatação para conectar fragmentos próximos de um mesmo objeto
    img_dilatada = cv2.dilate(img_filtrada, kernel, iterations=iteracoes_dilatacao)

    # 4. Realizar a contagem final das componentes pós-processamento
    num_larvas_final, _ = cv2.connectedComponents(img_dilatada)
    contagem_final = num_larvas_final - 1 # Descontando o rótulo do fundo

    # Exibir métricas e resultados visuais
    print(f"--- ANÁLISE INTERATIVA: X = {X} | DILATAÇÕES = {iteracoes_dilatacao} ---")
    print(f"Contagem Obtida: {contagem_final} objetos")
    print("-" * 50)

    mostrar(img_filtrada, f'1. Filtrada (Sem componentes < {X} px)')
    mostrar(img_dilatada, f'2. Dilatada ({iteracoes_dilatacao}x) - Resultado p/ Contagem')

# Geração dos controles deslizantes para calibração em tempo real no Google Colab
interact(analise_interativa,
         X=IntSlider(min=0, max=100, step=5, value=50, description='Tamanho X:'),
         iteracoes_dilatacao=IntSlider(min=0, max=5, step=1, value=2, description='Dilatações:'));

## Etapa 5: Processamento Consolidado e Contagem Final
Após a fase de testes, consolidamos o pipeline com os parâmetros ideais identificados: filtro de tamanho mínimo **$X = 50$ pixels** e **2 iterações de dilatação**.

O código varre a imagem limpa, elimina os ruídos menores que o limiar, une os fragmentos das larvas e executa o algoritmo de rotulação de componentes conexas. Cada grupo isolado de pixels brancos recebe um número de identificação único (ID).

O resultado final exibe a foto original ao lado do mapa de contagem, onde cada larva identificada é pintada com uma cor diferente, entregando a estimativa final automatizada com altíssima precisão.

In [ ]:
X_otimo = 5
dilatacoes_otimas = 2

kernel = np.ones((3,3), np.uint8)

num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(imagem_binaria, connectivity=8)

img_filtrada = np.zeros_like(imagem_binaria)
for i in range(1, num_labels):
    area = stats[i, cv2.CC_STAT_AREA]
    if area >= X_otimo:
        img_filtrada[labels == i] = 255

img_final_contagem = cv2.dilate(img_filtrada, kernel, iterations=dilatacoes_otimas)

num_larvas_final, labels_finais = cv2.connectedComponents(img_final_contagem)
contagem_final = num_larvas_final - 1

print("="*40)
print(f"   RELATÓRIO DE CONTAGEM DE PÓS-LARVAS")
print("="*40)
print(f"Parâmetro de Filtro (X): {X_otimo} pixels")
print(f"Quantidade de Dilatações: {dilatacoes_otimas}")
print(f"CONTAGEM OBTIDA: {contagem_final} larvas")
print("="*40)

fig, axs = plt.subplots(1, 2, figsize=(18, 8))

# Original
axs[0].imshow(cv2.cvtColor(bgr_original, cv2.COLOR_BGR2RGB))
axs[0].set_title('Foto Original Recortada/Carregada')
axs[0].axis('off')

# Resultado com os IDs coloridos
axs[1].imshow(labels_finais, cmap='jet')
axs[1].set_title(f'Resultado da Contagem ({contagem_final} detectadas)')
axs[1].axis('off')

plt.tight_layout()
plt.show()